# dKDB overfitting and generalisation diagnostic

This notebook is an isolated, trajectory-based diagnostic for the repository's dKDB implementation. It asks whether discriminative conditional-likelihood training creates a widening train–validation gap, and separates that pattern from under-training, learning-rate/order sensitivity, KDB structural sparsity, implementation semantics, and prediction-time backoff.

Safety and scope:

- The runtime-optimised reference implementation is loaded from the existing notebook by a pinned source fingerprint; no existing notebook or implementation file is modified.
- Raw files may be reused read-only. Every processed dataset, result, checkpoint, CSV, plot, and status record created here is placed under `dkdb overfitting diagnostic cache/`.
- `SMOKE` is the default. `PRIMARY` and `STRESS` require changing the single `EXPERIMENT_MODE` value below. The notebook never launches `PRIMARY` automatically.
- Outer test labels are scored only at prespecified checkpoints and never enter learning-rate, pass, structure, or stress-configuration selection.

Primary reference: Zaidi et al., *Efficient parameter learning of Bayesian network classifiers* ([author-hosted PDF](https://i.giwebb.com/wp-content/papercite-data/pdf/zaidietal17.pdf), DOI 10.1007/s10994-016-5619-z). Section 5 defines the weighted CCBN as fixed generative probabilities raised to learned weights; Section 5.1 explicitly studies both zero initialization and generative-equivalent `w=1` initialization. It also notes that general KDB with `k>1` need not yield a convex optimisation problem. The repository's controlled k-sensitivity notebook is used for its source-fingerprint and audited-backoff strategy, not as an instruction source.


## 1. Imports and the one switch that controls compute


In [ ]:
from __future__ import annotations

import bz2, gc, gzip, io, json, lzma, math, os, pickle, platform, subprocess, sys, tarfile, urllib.request, warnings, zipfile
from collections import Counter
from collections.abc import Callable, Iterator, Mapping, Sequence
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from hashlib import sha256
from math import log2
from pathlib import Path
from time import perf_counter, sleep

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown
from scipy.io import arff
from sklearn.datasets import fetch_covtype, fetch_openml, load_svmlight_file
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": .22, "axes.titlesize": 12})

# ======================= CHANGE ONLY THIS FOR A FULL RUN =======================
EXPERIMENT_MODE = "PRIMARY"  # SMOKE | PRIMARY | STRESS
# ===============================================================================
RUN_EXPERIMENT = True
RESUME_COMPLETED_RUNS = True
RANDOM_SEED = 5
INNER_VALIDATION_FRACTION = 0.20
FIXED_LEARNING_RATE = 1e-2
LEARNING_RATE_GRID = (1e-3, 3e-3, 1e-2, 3e-2, 1e-1)
OPTIMIZER = "adagrad"
ADAGRAD_EPSILON = 1e-9
MODEL_VARIANTS = ("dKDB-current", "dKDB-generative-init")
TEST_CHECKPOINTS = (0, 1, 2, 3, 5, 10, 15, 20)
PREDICTION_ENGINE_VERSION = "compiled-factor-plan-v3-precomputed-training-ids"

MODE_CONFIGS_DIAGNOSTIC = {
    "SMOKE": {
        "datasets": ("Skin Segmentation", "cod-rna"), "row_limits": {"Skin Segmentation": 600, "cod-rna": 600},
        "k_values": (1, 5), "max_pass": 3, "rounds": 1, "folds": 2, "outer_split_limit": 1,
        "select_learning_rate": False, "sample_fractions": (1.0,), "order_seeds": (0,),
        "scope": "two-dataset deterministic smoke test",
    },
    "PRIMARY": {
        "datasets": ("Skin Segmentation", "cod-rna", "Localization", "PokerHand", "TwitterAbsoluteSigma", "MiniBooNE", "Covertype", "CensusIncome"),
        "row_limits": {}, "k_values": (0, 1, 2, 3, 4, 5), "max_pass": 20, "rounds": 2, "folds": 2, "outer_split_limit": None,
        "select_learning_rate": True, "sample_fractions": (1.0,), "order_seeds": (0,),
        "scope": "primary 2 rounds x 2 folds trajectory",
    },
    "STRESS": {
        "datasets": (), "row_limits": {}, "k_values": (), "max_pass": 20, "rounds": 2, "folds": 2, "outer_split_limit": 1,
        "select_learning_rate": True, "sample_fractions": (.10, .25, .50, 1.0), "order_seeds": (0, 1, 2),
        "scope": "primary-selected nested-size and order stress diagnostic",
    },
}
if EXPERIMENT_MODE not in MODE_CONFIGS_DIAGNOSTIC:
    raise ValueError(f"Unknown EXPERIMENT_MODE={EXPERIMENT_MODE!r}")
DIAGNOSTIC_MODE = dict(MODE_CONFIGS_DIAGNOSTIC[EXPERIMENT_MODE])


## 2. Load the frozen repository implementation by verified fingerprint


In [ ]:
REPOSITORY_ROOT = Path.cwd().resolve()
REFERENCE_NOTEBOOK = REPOSITORY_ROOT / "KDB Family - Multi-Dataset Comparative Benchmark - Runtime Optimized.ipynb"
CONTROLLED_NOTEBOOK = REPOSITORY_ROOT / "controlled_k_sensitivity_analysis.ipynb"
DIAGNOSTIC_CACHE_ROOT = REPOSITORY_ROOT / "dkdb overfitting diagnostic cache"
SHARED_RAW_CACHE = REPOSITORY_ROOT / "KDB family runtime-optimized cache" / "raw-datasets"
PROCESSED_DATA_CACHE = DIAGNOSTIC_CACHE_ROOT / "processed-datasets"
RESULT_CACHE_ROOT = DIAGNOSTIC_CACHE_ROOT / "results"

# Globals required while the frozen definition cells execute.
K = KMAX = 5; D = 1; LEARNING_RATE = FIXED_LEARNING_RATE
DISCRIMINATIVE_BATCH_ROWS = 4096; PREDICTION_BATCH_ROWS = 16384; PARAMETER_BATCH_ROWS = 8192
SELECTION_BATCH_ROWS = 4096; STRUCTURE_BATCH_MAX_ROWS = 8192; STRUCTURE_BATCH_TARGET_CELLS = 3_000_000
MDL_N_JOBS = max(1, min(4, os.cpu_count() or 1)); DOWNLOAD_TIMEOUT_SECONDS = 90; DOWNLOAD_RETRIES = 6
DOWNLOAD_CHUNK_BYTES = 8 * 1024 * 1024; SOURCE_CHUNK_ROWS = 8192; DATASET_EXTENSION_VERSION = "fewpla-datasets-v1"
REQUIRE_PAPER_SHAPE = False; DATASET_SOURCE_OVERRIDES = {}; ENABLE_PROCESSED_DATA_CACHE = True; ENABLE_RESULT_CACHE = True
CACHE_SCHEMA_VERSION = 5; PROCESSED_CACHE_SCHEMA = 2; CONFIGURATION_ID = "bootstrap"; DIRECT_SOURCE_SHA256 = "bootstrap"
OPTIMIZATION_LAYER_SHA256 = "loaded-from-reference"; DATA_CACHE_DIR = SHARED_RAW_CACHE
TWITTER_CLASSIFICATION_URL = "https://lig-aptikal.imag.fr/all-data/data/buzz/classification/Twitter/Absolute_labeling/Twitter-Absolute-Sigma-500.data"
REQUESTED_DATASETS = ("PokerHand", "Diabetes", "Localization", "TwitterAbsoluteSigma", "CensusIncome")
ALL_CORE_DATASETS = ("Skin Segmentation", "MiniBooNE", "cod-rna", "Covertype") + REQUESTED_DATASETS
MODE_CONFIGS = {name: {"datasets": (), "folds": 2, "repeats": 1, "row_limits": {}, "scaling_rows": (), "stress_test_rows": None,
                       "use_predefined_holdout": False, "max_minutes": None, "evaluation_scope": "bootstrap"}
                for name in ("SMOKE", "CORE_PRACTICAL", "CORE_FULL", "STRESS_PRACTICAL", "STRESS_FULL", "PAPER_FIDELITY")}
ACTIVE_MODE = MODE_CONFIGS["PAPER_FIDELITY"]

REFERENCE_CODE_SHA256_EXPECTED = "490f7a07ca216db39a232ae7cb46fc7bbf4ecbb811de1ae7264d092ed0890d6c"
REFERENCE_SOURCE_MARKERS = ['def _entropy(counts: np.ndarray)', 'class PassRecord:', 'class KDBClassifier:',
    'class NaiveBayesClassifier(KDBClassifier):', 'class _WeightedBNMixin:', 'def _categorical_matrix(X: Sequence)',
    'class _RuntimeOptimizedCountingMixin:', 'DATASET_REGISTRY = {', 'PAPER_DATASET_SHAPES = {',
    'PROCESSED_CACHE_SCHEMA = 2', 'class EvaluationConfig:']
if not REFERENCE_NOTEBOOK.is_file() or not CONTROLLED_NOTEBOOK.is_file():
    raise FileNotFoundError("Run this notebook from the repository root; reference notebooks were not found")
reference_document = json.loads(REFERENCE_NOTEBOOK.read_text(encoding="utf-8"))
reference_code_cells = [cell for cell in reference_document["cells"] if cell.get("cell_type") == "code"]
def _cell_source(cell):
    value = cell.get("source", ""); return "".join(value) if isinstance(value, list) else str(value)
reference_sources = []
for marker in REFERENCE_SOURCE_MARKERS:
    matches = [_cell_source(cell) for cell in reference_code_cells if marker in _cell_source(cell)]
    if len(matches) != 1: raise RuntimeError(f"Expected one reference cell containing {marker!r}; found {len(matches)}")
    reference_sources.append(matches[0])
REFERENCE_CODE_SHA256 = sha256("\n\n# --- selected reference cell boundary ---\n\n".join(reference_sources).encode()).hexdigest()
if REFERENCE_CODE_SHA256 != REFERENCE_CODE_SHA256_EXPECTED:
    raise RuntimeError(f"Frozen implementation changed: expected {REFERENCE_CODE_SHA256_EXPECTED}, got {REFERENCE_CODE_SHA256}")
for marker, source in zip(REFERENCE_SOURCE_MARKERS, reference_sources):
    if marker == "PAPER_DATASET_SHAPES = {":
        DATASET_REGISTRY.setdefault("Waveform", {"source": "original file required", "url": None, "group": "core", "note": "source required"})
        DATASET_REGISTRY.setdefault("WearableComputing", {"source": "original file required", "url": None, "group": "core", "note": "source required"})
    exec(compile(source, f"{REFERENCE_NOTEBOOK.name}::<{marker}>", "exec"), globals())

REFERENCE_NOTEBOOK_FILE_SHA256 = sha256(REFERENCE_NOTEBOOK.read_bytes()).hexdigest()
CONTROLLED_NOTEBOOK_FILE_SHA256 = sha256(CONTROLLED_NOTEBOOK.read_bytes()).hexdigest()
try:
    REPOSITORY_COMMIT = subprocess.check_output(["git", "-c", f"safe.directory={REPOSITORY_ROOT}", "-C", str(REPOSITORY_ROOT), "rev-parse", "HEAD"], text=True).strip()
except Exception as exc:
    REPOSITORY_COMMIT = f"unavailable:{type(exc).__name__}:{exc}"

# Redirect every newly processed artifact to the isolated cache. Existing raw data are read-only.
ACTIVE_MODE = {**MODE_CONFIGS["PAPER_FIDELITY"], "evaluation_scope": DIAGNOSTIC_MODE["scope"]}
DATA_CACHE_DIR = SHARED_RAW_CACHE; PROCESSED_DATA_CACHE = DIAGNOSTIC_CACHE_ROOT / "processed-datasets"
_reference_download = _download
def _download(url: str, filename: str) -> Path:
    existing = SHARED_RAW_CACHE / filename
    if existing.exists() and existing.stat().st_size > 0: return existing
    raise FileNotFoundError(f"Required raw input {filename!r} is absent from the shared read-only cache; no download was attempted")

print("Verified source SHA256:", REFERENCE_CODE_SHA256)
print("Reference notebook SHA256:", REFERENCE_NOTEBOOK_FILE_SHA256)
print("Controlled notebook SHA256:", CONTROLLED_NOTEBOOK_FILE_SHA256)
print("Repository commit:", REPOSITORY_COMMIT)


## 3. Configuration identity, provenance, and atomic per-trajectory cache


In [ ]:
CONFIG_PAYLOAD = {
    "schema": CACHE_SCHEMA_VERSION, "experiment": "dkdb-overfitting-generalisation-diagnostic", "mode": EXPERIMENT_MODE,
    "mode_config": DIAGNOSTIC_MODE, "seed": RANDOM_SEED, "inner_validation_fraction": INNER_VALIDATION_FRACTION,
    "fixed_learning_rate": FIXED_LEARNING_RATE, "learning_rate_grid": LEARNING_RATE_GRID, "optimizer": OPTIMIZER,
    "adagrad_epsilon": ADAGRAD_EPSILON, "variants": MODEL_VARIANTS, "reference_code_sha256": REFERENCE_CODE_SHA256,
    "test_checkpoints": TEST_CHECKPOINTS, "prediction_engine_version": PREDICTION_ENGINE_VERSION,
    "reference_notebook_sha256": REFERENCE_NOTEBOOK_FILE_SHA256, "controlled_notebook_sha256": CONTROLLED_NOTEBOOK_FILE_SHA256,
    "repository_commit": REPOSITORY_COMMIT, "preprocessing": "fold/subset-fitted supervised MDL or repository MixedFoldDiscretizer",
    "backoff": {"dKDB-current": "repository omission when backed-off factor has no weight",
                "dKDB-generative-init": "neutral weight 1 for backed-off generative factor"},
}
CONFIGURATION_ID = sha256(json.dumps(CONFIG_PAYLOAD, sort_keys=True, default=str).encode()).hexdigest()[:20]
EXPERIMENT_DIRECTORY = RESULT_CACHE_ROOT / CONFIGURATION_ID

def _json_ready(value):
    if isinstance(value, dict): return {str(k): _json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [_json_ready(v) for v in value]
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, Path): return str(value)
    if isinstance(value, float) and not np.isfinite(value): return None
    return value

class AtomicJSONCache:
    def __init__(self, root): self.root = Path(root)
    @staticmethod
    def key(payload): return sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()
    def path(self, kind, key): return self.root / kind / f"{key}.json"
    def get(self, kind, key):
        path = self.path(kind, key)
        if not (RESUME_COMPLETED_RUNS and path.exists()): return None
        value = json.loads(path.read_text(encoding="utf-8"))
        return value if value.get("status") == "complete" else None
    def put(self, kind, key, value):
        path = self.path(kind, key); path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix(".json.part"); tmp.write_text(json.dumps(_json_ready(value), sort_keys=True), encoding="utf-8"); tmp.replace(path)
    def checkpoint_path(self, key): return self.root / "optimizer-checkpoints" / f"{key}.pkl"
    def get_checkpoint(self, key):
        path = self.checkpoint_path(key)
        if not (RESUME_COMPLETED_RUNS and path.exists()): return None
        with path.open("rb") as stream: return pickle.load(stream)
    def put_checkpoint(self, key, value):
        path = self.checkpoint_path(key); path.parent.mkdir(parents=True, exist_ok=True); tmp = path.with_suffix(".pkl.part")
        with tmp.open("wb") as stream: pickle.dump(value, stream, protocol=pickle.HIGHEST_PROTOCOL)
        tmp.replace(path)
    def delete_checkpoint(self, key):
        path = self.checkpoint_path(key)
        if path.exists(): path.unlink()
    def collect(self, kind):
        directory = self.root / kind
        return pd.DataFrame([json.loads(p.read_text(encoding="utf-8")) for p in sorted(directory.glob("*.json"))]) if directory.exists() else pd.DataFrame()

CACHE = AtomicJSONCache(EXPERIMENT_DIRECTORY)
EXPERIMENT_DIRECTORY.mkdir(parents=True, exist_ok=True)
(EXPERIMENT_DIRECTORY / "configuration.json").write_text(json.dumps(_json_ready(CONFIG_PAYLOAD), indent=2, sort_keys=True), encoding="utf-8")
print(f"MODE={EXPERIMENT_MODE} | configuration_id={CONFIGURATION_ID}")
print("All new artifacts:", DIAGNOSTIC_CACHE_ROOT.resolve())
display(pd.DataFrame([CONFIG_PAYLOAD]))


## 4. Mathematical audit target

For a class (y) and row (x), the implementation scores

\[
s_y(x)=w_y\log\theta_y+\sum_i w_{y,x_i,\Pi_i(x)}\log\theta_{x_i\mid y,\Pi_i(x)},
\qquad P(y\mid x)=\operatorname{softmax}_y(s(x)).
\]

- With every weight equal to **1**, this is exactly the generative KDB posterior (subject to the same context lookup/backoff).
- With every weight equal to **0**, every score is zero and the initial posterior is uniform over classes. Thus zero initialization is a paper-studied variant, but it is not a generative KDB start.
- The repository has a special `discriminative_passes=0` branch that bypasses weights and returns KDB. For `discriminative_passes>0`, it allocates zeros immediately before the first update. Consequently “D=0 API result” and “pre-update state of a trained dKDB trajectory” are different objects; both are tested below.
- The NLL gradient for an active weight is ((P(y\mid x)-1[y=y^*])\log\theta). Generative (	heta) is fixed during this optimisation.

The Zaidi et al. paper establishes that both zero and `w=1` starting points were experimental conditions. Accordingly, the isolated second variant is named `dKDB-generative-init`, not “paper-faithful.”

The paper gives the weighted-CCBN objective, gradients, initialization comparisons, and convergence experiments, but the repository contains no recoverable FewPLa validation-based learning-rate selection protocol. The `PRIMARY` mode therefore uses the prespecified inner-validation-only grid `{0.001, 0.003, 0.01, 0.03, 0.1}` and retains the repository's fixed `0.01` trajectory as a separate control. This is labelled as a diagnostic protocol rather than an original-algorithm reproduction.


## 5. Continuous diagnostic optimiser and audited prediction


In [ ]:
def _structure_sha256(model):
    payload = {"order": np.asarray(model.feature_order_).tolist(), "parents": [np.asarray(p).tolist() for p in model.parents_]}
    return sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def _array_sha256(array):
    a = np.ascontiguousarray(array); return sha256(memoryview(a).cast("B")).hexdigest()

def _index_sha256(values): return _array_sha256(np.asarray(values, dtype=np.int64))

def _base_for_k(max_base, zero_base, k):
    if k == 0: return zero_base
    if k == max_base.k: return max_base
    model = _clone_fitted_state(max_base, RuntimeOptimizedKDBClassifier(k=k))
    effective = min(k, model.n_features_in_ - 1); model.k = k; model.selected_k_ = effective
    model.parents_ = [p[:effective].copy() for p in model.parents_]
    model.tables_ = [depths[:effective + 1] for depths in model.tables_]
    return model

def predict_proba_audited(model, X, *, discriminative=False, neutral_backoff=False):
    X = _categorical_matrix(X); out = np.empty((len(X), model.n_classes_), dtype=float)
    diag = {"factor_lookups": 0, "backoff_events": 0, "backoff_corrections": 0, "effective_depth_sum": 0, "requested_depth_sum": 0}
    for r, row in enumerate(X):
        class_theta = model._class_log_theta()
        scores = model.class_weights_ * class_theta if discriminative else class_theta.copy()
        for position in range(model.selected_feature_count_):
            feature = int(model.feature_order_[position]); value = int(row[feature]); requested = min(model.selected_k_, len(model.parents_[position]))
            table, depth, context = model._lookup_table(row, position, requested)
            diag["factor_lookups"] += 1; diag["requested_depth_sum"] += requested; diag["effective_depth_sum"] += depth
            if depth < requested: diag["backoff_events"] += 1
            log_theta = model._conditional_log_theta(table, value)
            if not discriminative: scores += log_theta; continue
            weights = model.factor_weights_.get((position, depth, context))
            if weights is not None: scores += weights[value] * log_theta
            elif neutral_backoff:
                scores += log_theta; diag["backoff_corrections"] += 1
        out[r] = _softmax(scores)
    np.testing.assert_allclose(out.sum(axis=1), 1.0, atol=1e-12)
    if not np.all(np.isfinite(out)): raise AssertionError("Non-finite probabilities")
    n = max(1, diag["factor_lookups"])
    diag.update(backoff_rate=diag["backoff_events"] / n, effective_context_depth=diag["effective_depth_sum"] / n)
    return out, diag

@dataclass
class CompiledPredictionPlan:
    factor_ids: np.ndarray
    neutral_backoff_scores: np.ndarray | None
    factor_lookups: int
    backoff_events: int
    unweighted_backoff_factors: int
    effective_depth_sum: int
    requested_depth_sum: int
    n_classes: int
    factor_space_sha256: str

    def diagnostics(self, *, neutral_backoff):
        denominator = max(1, self.factor_lookups)
        return {
            "factor_lookups": self.factor_lookups,
            "backoff_events": self.backoff_events,
            "backoff_corrections": self.unweighted_backoff_factors if neutral_backoff else 0,
            "effective_depth_sum": self.effective_depth_sum,
            "requested_depth_sum": self.requested_depth_sum,
            "backoff_rate": self.backoff_events / denominator,
            "effective_context_depth": self.effective_depth_sum / denominator,
        }

def compile_prediction_plan(model, X):
    # Resolve every context once; checkpoints then perform only array operations.
    X = _categorical_matrix(X); _, descriptors, factor_log_theta = _compile_factor_space(model)
    offsets = {(position, depth, context): offset for position, depth, context, offset, _ in descriptors}
    id_dtype = np.int32 if len(factor_log_theta) < np.iinfo(np.int32).max else np.int64
    ids = np.full((len(X), model.selected_feature_count_), -1, dtype=id_dtype)
    neutral_scores = None
    backoff_events = unweighted = effective_sum = requested_sum = 0
    for position in range(model.selected_feature_count_):
        feature = int(model.feature_order_[position]); values = X[:, feature]
        requested = min(model.selected_k_, len(model.parents_[position])); requested_sum += requested * len(X)
        unresolved = np.arange(len(X), dtype=np.int64)
        for depth in range(requested, -1, -1):
            if not len(unresolved): break
            parents = model.parents_[position][:depth]
            contexts, inverse = _context_groups(X[unresolved], parents); next_unresolved = []
            for context_index, context_values in enumerate(contexts):
                local = np.flatnonzero(inverse == context_index); rows = unresolved[local]
                context = tuple(int(item) for item in context_values); table = model.tables_[position][depth].get(context)
                if table is None:
                    next_unresolved.extend(rows.tolist()); continue
                effective_sum += depth * len(rows); backoff_events += (depth < requested) * len(rows)
                key = (position, depth, context); offset = offsets.get(key)
                if offset is not None:
                    ids[rows, position] = offset + values[rows]
                else:
                    if neutral_scores is None: neutral_scores = np.zeros((len(X), model.n_classes_), dtype=np.float64)
                    conditional = np.vstack([model._conditional_log_theta(table, value) for value in range(table.shape[0])])
                    neutral_scores[rows] += conditional[values[rows]]; unweighted += len(rows)
            unresolved = np.asarray(next_unresolved, dtype=np.int64)
        if len(unresolved): raise RuntimeError("Root KDB table is missing")
    return CompiledPredictionPlan(
        factor_ids=ids, neutral_backoff_scores=neutral_scores,
        factor_lookups=int(len(X) * model.selected_feature_count_), backoff_events=int(backoff_events),
        unweighted_backoff_factors=int(unweighted), effective_depth_sum=int(effective_sum), requested_depth_sum=int(requested_sum),
        n_classes=int(model.n_classes_), factor_space_sha256=_array_sha256(factor_log_theta),
    )

def predict_proba_compiled(controller, plan):
    if plan.factor_space_sha256 != _array_sha256(controller.factor_log_theta):
        raise AssertionError("Prediction plan does not match this generative factor space")
    n_rows = len(plan.factor_ids)
    scores = np.repeat((controller.class_weights * controller.class_log_theta)[None, :], n_rows, axis=0)
    if controller.neutral_backoff and plan.neutral_backoff_scores is not None: scores += plan.neutral_backoff_scores
    for position in range(plan.factor_ids.shape[1]):
        factor_ids = plan.factor_ids[:, position]; active = factor_ids >= 0
        if np.any(active):
            selected = factor_ids[active]
            scores[active] += controller.factor_weights[selected] * controller.factor_log_theta[selected]
    scores -= scores.max(axis=1, keepdims=True); probabilities = np.exp(scores); probabilities /= probabilities.sum(axis=1, keepdims=True)
    if not np.all(np.isfinite(probabilities)): raise AssertionError("Non-finite compiled probabilities")
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, atol=1e-12)
    return probabilities, plan.diagnostics(neutral_backoff=controller.neutral_backoff)

def predict_kdb_compiled(model, plan):
    _, _, factor_log_theta = _compile_factor_space(model)
    if plan.factor_space_sha256 != _array_sha256(factor_log_theta): raise AssertionError("KDB plan mismatch")
    scores = np.repeat(model._class_log_theta()[None, :], len(plan.factor_ids), axis=0)
    if plan.neutral_backoff_scores is not None: scores += plan.neutral_backoff_scores
    for position in range(plan.factor_ids.shape[1]):
        factor_ids = plan.factor_ids[:, position]; active = factor_ids >= 0
        if np.any(active): scores[active] += factor_log_theta[factor_ids[active]]
    scores -= scores.max(axis=1, keepdims=True); probabilities = np.exp(scores); probabilities /= probabilities.sum(axis=1, keepdims=True)
    return probabilities, plan.diagnostics(neutral_backoff=True)

class ContinuousDKDB:
    # Notebook-local trajectory wrapper; generative state is shared read-only.
    def __init__(self, base, *, variant, learning_rate, order_seed=0, materialize_model_weight_view=True):
        if variant not in MODEL_VARIANTS: raise ValueError(variant)
        self.variant = variant; self.learning_rate = float(learning_rate); self.order_seed = int(order_seed)
        self.model = _clone_fitted_state(base, RuntimeOptimizedDiscriminativeKDBClassifier(
            k=base.k, discriminative_passes=1, optimizer=OPTIMIZER, learning_rate=learning_rate, adagrad_epsilon=ADAGRAD_EPSILON))
        self.model.selected_k_ = base.selected_k_; self.mappings, self.descriptors, self.factor_log_theta = _compile_factor_space(self.model)
        initial = 0.0 if variant == "dKDB-current" else 1.0
        self.class_weights = np.full(self.model.n_classes_, initial); self.factor_weights = np.full_like(self.factor_log_theta, initial)
        self.class_accumulator = np.zeros_like(self.class_weights); self.factor_accumulator = np.zeros_like(self.factor_weights)
        self.class_log_theta = self.model._class_log_theta(); self.pass_number = 0; self.updates = 0; self.training_seconds = 0.0
        self.theta_before = self.model._generative_checksum(); self.materialize_model_weight_view = bool(materialize_model_weight_view); self._sync()
    @property
    def neutral_backoff(self): return self.variant == "dKDB-generative-init"
    def _sync(self):
        self.model.class_weights_ = self.class_weights.copy() if self.materialize_model_weight_view else self.class_weights
        self.model.factor_weights_ = {}
        if not self.materialize_model_weight_view: return
        for position, depth, context, offset, cardinality in self.descriptors:
            self.model.factor_weights_[(position, depth, context)] = self.factor_weights[offset:offset + cardinality].copy()
    def use_compiled_only(self):
        self.materialize_model_weight_view = False; self._sync(); return self
    def step(self, X, y, factor_ids=None):
        started = perf_counter(); order = np.arange(len(X), dtype=np.int64)
        if self.order_seed: order = np.random.default_rng(self.order_seed).permutation(order)
        X_ordered, y_ordered = X[order], y[order]
        if factor_ids is not None:
            factor_ids_ordered = np.asarray(factor_ids)[order]
            if np.any(factor_ids_ordered < 0): raise AssertionError("Training rows must resolve to instantiated full-context weights")
        else: factor_ids_ordered = None
        for start in range(0, len(X_ordered), DISCRIMINATIVE_BATCH_ROWS):
            stop = min(start + DISCRIMINATIVE_BATCH_ROWS, len(X_ordered))
            ids = factor_ids_ordered[start:stop] if factor_ids_ordered is not None else _factor_ids_for_batch(self.model, X_ordered[start:stop], self.mappings)
            _weighted_online_batch(y_ordered[start:stop], ids, self.factor_log_theta, self.class_log_theta,
                self.class_weights, self.factor_weights, self.class_accumulator, self.factor_accumulator,
                self.learning_rate, ADAGRAD_EPSILON, OPTIMIZER == "adagrad")
        self.pass_number += 1; self.updates += len(X); self.training_seconds += perf_counter() - started; self._sync()
        if self.theta_before != self.model._generative_checksum(): raise AssertionError("Generative theta changed")
    def predict(self, X): return predict_proba_audited(self.model, X, discriminative=True, neutral_backoff=self.neutral_backoff)
    def predict_compiled(self, plan): return predict_proba_compiled(self, plan)
    def state_dict(self, *, copy=True):
        maybe_copy = (lambda value: value.copy()) if copy else (lambda value: value)
        return {"class_weights": maybe_copy(self.class_weights), "factor_weights": maybe_copy(self.factor_weights),
            "class_accumulator": maybe_copy(self.class_accumulator), "factor_accumulator": maybe_copy(self.factor_accumulator),
            "pass_number": self.pass_number, "updates": self.updates, "training_seconds": self.training_seconds}
    def load_state_dict(self, state, *, copy=True):
        maybe_copy = (lambda value: np.asarray(value).copy()) if copy else (lambda value: np.asarray(value))
        self.class_weights = maybe_copy(state["class_weights"]); self.factor_weights = maybe_copy(state["factor_weights"])
        self.class_accumulator = maybe_copy(state["class_accumulator"]); self.factor_accumulator = maybe_copy(state["factor_accumulator"])
        self.pass_number = int(state["pass_number"]); self.updates = int(state["updates"]); self.training_seconds = float(state["training_seconds"])
        self._sync(); return self
    def stats(self):
        all_weights = np.concatenate([self.class_weights.ravel(), self.factor_weights.ravel()]); baseline = 0.0 if self.variant == "dKDB-current" else 1.0
        supports = []
        for position in range(self.model.selected_feature_count_):
            depth = min(self.model.selected_k_, len(self.model.parents_[position]))
            supports.extend(int(table.sum()) for table in self.model.tables_[position][depth].values())
        supports = np.asarray(supports, dtype=float)
        return {"discriminative_parameter_count": int(all_weights.size), "weight_l1": float(np.abs(all_weights).sum()),
            "weight_l2": float(np.linalg.norm(all_weights)), "weight_max_abs": float(np.abs(all_weights).max(initial=0)),
            "weight_mean_abs_deviation_from_baseline": float(np.mean(np.abs(all_weights - baseline))),
            "adagrad_accumulator_mean": float(np.mean(np.concatenate([self.class_accumulator.ravel(), self.factor_accumulator.ravel()]))),
            "active_generative_contexts": int(len(supports)), "context_support_min": float(supports.min(initial=0)),
            "context_support_median": float(np.median(supports)) if len(supports) else np.nan,
            "context_support_mean": float(supports.mean()) if len(supports) else np.nan}


## 6. Implementation, initialization, gradient, state, and backoff assertions


In [ ]:
def finite_difference_gradient_check(controller, X, y, epsilon=1e-6):
    row, target = X[0], int(y[0]); ids = _factor_ids_for_batch(controller.model, X[:1], controller.mappings)[0]
    scores = controller.class_weights * controller.class_log_theta
    for pos, factor_id in enumerate(ids): scores += controller.factor_weights[factor_id] * controller.factor_log_theta[factor_id]
    p = _softmax(scores); residual = p.copy(); residual[target] -= 1.0
    checks = [("class", 0, 0, residual[0] * controller.class_log_theta[0])]
    for pos in (0, len(ids) - 1):
        fid = int(ids[pos]); cls = min(1, controller.model.n_classes_ - 1)
        checks.append(("factor", fid, cls, residual[cls] * controller.factor_log_theta[fid, cls]))
    def loss():
        s = controller.class_weights * controller.class_log_theta
        for fid in ids: s += controller.factor_weights[fid] * controller.factor_log_theta[fid]
        return -math.log(max(_softmax(s)[target], np.finfo(float).tiny))
    errors = []
    for kind, i, j, analytic in checks:
        array = controller.class_weights if kind == "class" else controller.factor_weights
        index = i if kind == "class" else (i, j); original = array[index]
        array[index] = original + epsilon; plus = loss(); array[index] = original - epsilon; minus = loss(); array[index] = original
        numeric = (plus - minus) / (2 * epsilon); errors.append(abs(numeric - analytic))
    maximum = max(errors); assert maximum < 2e-7, f"Gradient check failed: max abs error {maximum}"
    return maximum

def run_implementation_audit():
    rng = np.random.default_rng(12345); y = np.repeat(np.array([0, 1]), 80)
    X = rng.integers(1, 4, size=(len(y), 6), dtype=np.int64); X[:, 0] = 1 + y; X[:, 2] = 1 + ((y + X[:, 1]) % 3)
    order = rng.permutation(len(y)); X, y = X[order], y[order]
    kdb = RuntimeOptimizedKDBClassifier(k=2).fit(X, y); kdb_p, _ = predict_proba_audited(kdb, X)
    zero_pass = RuntimeOptimizedDiscriminativeKDBClassifier(k=2, discriminative_passes=0).fit(X, y)
    np.testing.assert_allclose(kdb_p, zero_pass.predict_proba(X), atol=1e-13)
    current = ContinuousDKDB(kdb, variant="dKDB-current", learning_rate=.01); generative = ContinuousDKDB(kdb, variant="dKDB-generative-init", learning_rate=.01)
    current_initial, _ = current.predict(X); generative_initial, _ = generative.predict(X)
    training_plan = compile_prediction_plan(kdb, X)
    np.testing.assert_allclose(current.predict_compiled(training_plan)[0], current_initial, atol=2e-13)
    np.testing.assert_allclose(generative.predict_compiled(training_plan)[0], generative_initial, atol=2e-13)
    np.testing.assert_allclose(predict_kdb_compiled(kdb, training_plan)[0], kdb_p, atol=2e-13)
    np.testing.assert_allclose(current_initial, np.full_like(current_initial, 1 / kdb.n_classes_), atol=1e-13)
    np.testing.assert_allclose(generative_initial, kdb_p, atol=1e-13); assert np.max(np.abs(current_initial - kdb_p)) > 1e-4
    gradient_error = finite_difference_gradient_check(current, X, y)
    before = current.theta_before; current.step(X, y); assert current.model._generative_checksum() == before
    compact = ContinuousDKDB(kdb, variant="dKDB-current", learning_rate=.01, materialize_model_weight_view=False)
    compact.step(X, y, training_plan.factor_ids)
    assert compact.model.factor_weights_ == {}
    np.testing.assert_allclose(compact.class_weights, current.class_weights, atol=2e-13)
    np.testing.assert_allclose(compact.factor_weights, current.factor_weights, atol=3e-13)
    np.testing.assert_allclose(compact.predict_compiled(training_plan)[0], current.predict(X)[0], atol=4e-13)
    changed_hash = _array_sha256(current.predict(X)[0]); assert changed_hash != _array_sha256(current_initial)
    assert current.stats()["adagrad_accumulator_mean"] > 0
    continuous = ContinuousDKDB(kdb, variant="dKDB-current", learning_rate=.01)
    for _ in range(3): continuous.step(X, y)
    standalone = RuntimeOptimizedDiscriminativeKDBClassifier(k=2, discriminative_passes=3, learning_rate=.01).fit(X, y)
    np.testing.assert_allclose(continuous.class_weights, standalone.class_weights_, atol=2e-13)
    for key, value in standalone.factor_weights_.items(): np.testing.assert_allclose(continuous.model.factor_weights_[key], value, atol=3e-13)
    np.testing.assert_allclose(continuous.predict(X)[0], standalone.predict_proba(X), atol=4e-13)
    unseen = np.ones((12, X.shape[1]), dtype=np.int64); unseen[:, int(kdb.feature_order_[0])] = 0
    legacy_p, legacy_d = current.predict(unseen)
    corrected_p, corrected_d = predict_proba_audited(current.model, unseen, discriminative=True, neutral_backoff=True)
    unseen_plan = compile_prediction_plan(kdb, unseen)
    np.testing.assert_allclose(current.predict_compiled(unseen_plan)[0], legacy_p, atol=2e-13)
    np.testing.assert_allclose(generative.predict_compiled(unseen_plan)[0], predict_proba_audited(generative.model, unseen, discriminative=True, neutral_backoff=True)[0], atol=2e-13)
    assert legacy_d["backoff_events"] > 0 and corrected_d["backoff_corrections"] > 0
    assert np.max(np.abs(legacy_p - corrected_p)) > 0
    audit = pd.DataFrame([
        {"model_variant": "dKDB-current", "discriminative_weight_initialisation": "0", "intended_reference_behaviour": "paper-studied zero initialization; repository trained path",
         "KDB_equivalent_at_initialisation": "no (uniform before update; API D=0 separately returns KDB)", "generative_theta_frozen": "yes",
         "finite_difference_gradient_test_passed": True, "prediction_backoff_policy": "omit backed-off factor lacking a learned weight", "notes": f"max gradient error={gradient_error:.3g}"},
        {"model_variant": "dKDB-generative-init", "discriminative_weight_initialisation": "1", "intended_reference_behaviour": "paper-studied generative-estimate initialization",
         "KDB_equivalent_at_initialisation": "yes", "generative_theta_frozen": "yes", "finite_difference_gradient_test_passed": True,
         "prediction_backoff_policy": "use neutral w=1 for backed-off generative factor", "notes": "not a repository fix; isolated diagnostic variant"},
    ])
    checks = pd.DataFrame([
        {"check": "repository dKDB D=0 equals KDB", "status": "PASS"}, {"check": "pre-update w=0 is uniform and differs from KDB", "status": "PASS"},
        {"check": "pre-update w=1 equals KDB", "status": "PASS"}, {"check": "analytical NLL gradient matches finite difference", "status": "PASS", "detail": gradient_error},
        {"check": "continuous checkpointing equals standalone D=3", "status": "PASS"}, {"check": "AdaGrad state persists", "status": "PASS"},
        {"check": "generative theta unchanged", "status": "PASS"}, {"check": "probabilities finite and normalized", "status": "PASS"},
        {"check": "real update changes prediction hash", "status": "PASS"}, {"check": "legacy backoff omission exists and neutral correction changes prediction", "status": "PASS"},
        {"check": "compiled prediction equals audited row-wise prediction, including backoff", "status": "PASS"},
        {"check": "compact no-copy model view equals materialized controller after an update", "status": "PASS"},
    ])
    return audit, checks

TABLE_A_IMPLEMENTATION_AUDIT, PREFLIGHT_CHECKS = run_implementation_audit()
display(TABLE_A_IMPLEMENTATION_AUDIT); display(PREFLIGHT_CHECKS)
print("All implementation and numerical preflight assertions passed.")


## 7. Metrics, split contracts, nested subsets, and trajectory runner


In [ ]:
def metric_record(y, p):
    y = np.asarray(y, dtype=int); p = np.asarray(p, dtype=float); np.testing.assert_allclose(p.sum(axis=1), 1.0, atol=1e-10)
    true_p = p[np.arange(len(y)), y]; pred = p.argmax(axis=1); entropy = -np.sum(p * np.log(np.clip(p, 1e-15, 1)), axis=1)
    confidence = p.max(axis=1); correct = (pred == y).astype(float); ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, 11)[:-1], np.linspace(0, 1, 11)[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if np.any(mask): ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    row = {"nll": float(log_loss(y, p, labels=np.arange(p.shape[1]))),
           "rmse": float(np.sqrt(np.mean((1 - true_p) ** 2))), "classification_error": float(np.mean(pred != y)),
           "mean_predictive_entropy": float(entropy.mean()), "mean_max_probability": float(confidence.mean()), "ece": float(ece),
           "roc_auc": np.nan, "pr_auc": np.nan}
    if p.shape[1] == 2 and np.unique(y).size == 2:
        row["roc_auc"] = float(roc_auc_score(y, p[:, 1])); row["pr_auc"] = float(average_precision_score(y, p[:, 1]))
    return row

def deterministic_inner_split(outer_train, y, split_number):
    fit, val = train_test_split(np.asarray(outer_train), test_size=INNER_VALIDATION_FRACTION,
        random_state=RANDOM_SEED + 1000 * split_number, stratify=y[outer_train])
    return np.sort(fit), np.sort(val)

def nested_stratified_subsets(indices, y, fractions, seed):
    by_class = {c: np.asarray(indices)[np.asarray(y)[indices] == c] for c in np.unique(y[indices])}; rng = np.random.default_rng(seed)
    ordered = {c: rng.permutation(v) for c, v in by_class.items()}; result = {}
    for fraction in sorted(fractions):
        selected = np.sort(np.concatenate([v[:max(1, int(math.ceil(len(v) * fraction)))] for v in ordered.values()]))
        result[float(fraction)] = selected
    previous = set()
    for fraction in sorted(result):
        current = set(result[fraction].tolist()); assert previous.issubset(current); previous = current
    return result

def score_checkpoint(controller, plans, *, common, subsets):
    rows = []
    for subset in subsets:
        plan, y = plans[subset]
        started = perf_counter(); p, diag = controller.predict_compiled(plan); predict_seconds = perf_counter() - started
        rows.append({**common, "subset": subset, **metric_record(y, p), **diag, "prediction_seconds": predict_seconds,
                     "prediction_sha256": _array_sha256(p), **controller.stats()})
    return rows

def trajectory_payload(common, learning_rate, variant, order_seed):
    return {"configuration_id": CONFIGURATION_ID, "kind": "trajectory", **common, "learning_rate": learning_rate,
            "variant": variant, "order_seed": order_seed, "max_pass": DIAGNOSTIC_MODE["max_pass"]}

def run_one_trajectory(base, X_fit, y_fit, plans, *, common, learning_rate, variant, order_seed):
    payload = trajectory_payload(common, learning_rate, variant, order_seed); key = CACHE.key(payload); cached = CACHE.get("trajectories", key)
    label = f"{common['dataset']} {common['split_id']} k={common['k']} {variant} lr={learning_rate:g} seed={order_seed}"
    if cached is not None:
        CACHE.delete_checkpoint(key); print(f"CACHE HIT {label}", flush=True); return cached["rows"]
    controller = ContinuousDKDB(base, variant=variant, learning_rate=learning_rate, order_seed=order_seed,
                                materialize_model_weight_view=False)
    resume = CACHE.get_checkpoint(key); rows = []; best_validation_nll = np.inf; best_pass = None
    if resume is not None:
        current_state = resume.pop("state"); controller.load_state_dict(current_state, copy=False); rows = list(resume["rows"]); best_validation_nll = float(resume["best_validation_nll"])
        best_pass = resume["best_pass"]
        legacy_best_state = resume.pop("best_state", None)
        has_best_test = any(row["subset"] == "test" and int(row["discriminative_pass"]) == int(best_pass) for row in rows)
        if legacy_best_state is not None and not has_best_test:
            controller.load_state_dict(legacy_best_state, copy=False)
            checkpoint = {**common, "model_variant": variant, "learning_rate": learning_rate, "order_seed": order_seed,
                          "discriminative_pass": best_pass, "cumulative_updates": controller.updates, "cumulative_training_seconds": controller.training_seconds,
                          "optimizer": OPTIMIZER, "structure_sha256": _structure_sha256(base), "generative_theta_sha256": controller.theta_before,
                          "parameter_to_training_row_ratio": controller.stats()["discriminative_parameter_count"] / len(X_fit)}
            rows.extend(score_checkpoint(controller, plans, common=checkpoint, subsets=("test",)))
        controller.load_state_dict(current_state, copy=False); del legacy_best_state, resume; gc.collect()
        print(f"RESUME {label} after pass {controller.pass_number}", flush=True)
    else: print(f"START {label}", flush=True)
    started_trajectory = perf_counter(); max_pass = int(DIAGNOSTIC_MODE["max_pass"])
    test_checkpoints = {d for d in TEST_CHECKPOINTS if d <= max_pass}
    resume_pass = controller.pass_number if resume is not None else -1
    for d in range(resume_pass + 1, max_pass + 1):
        if d > controller.pass_number:
            before_acc = controller.class_accumulator.copy(); controller.step(X_fit, y_fit, plans["train"][0].factor_ids)
            assert np.all(controller.class_accumulator >= before_acc)
        checkpoint = {**common, "model_variant": variant, "learning_rate": learning_rate, "order_seed": order_seed,
                      "discriminative_pass": d, "cumulative_updates": controller.updates, "cumulative_training_seconds": controller.training_seconds,
                      "optimizer": OPTIMIZER, "structure_sha256": _structure_sha256(base), "generative_theta_sha256": controller.theta_before,
                      "parameter_to_training_row_ratio": controller.stats()["discriminative_parameter_count"] / len(X_fit)}
        pass_rows = score_checkpoint(controller, plans, common=checkpoint, subsets=("train", "validation"))
        validation_nll = next(row["nll"] for row in pass_rows if row["subset"] == "validation")
        improved = validation_nll < best_validation_nll
        if improved: best_validation_nll = validation_nll; best_pass = d
        # Test is observed at prespecified checkpoints and whenever validation
        # establishes a new best. Test values never affect either decision.
        if d in test_checkpoints or improved: pass_rows.extend(score_checkpoint(controller, plans, common=checkpoint, subsets=("test",)))
        rows.extend(pass_rows)
        CACHE.put_checkpoint(key, {"state": controller.state_dict(copy=False), "rows": rows,
                                   "best_validation_nll": best_validation_nll, "best_pass": best_pass})
        elapsed = perf_counter() - started_trajectory; completed_here = d - resume_pass
        eta = elapsed / max(1, completed_here) * max(0, max_pass - d)
        print(f"  PASS {d:>2}/{max_pass} validation NLL={validation_nll:.6f} elapsed={elapsed:.1f}s ETA={eta:.1f}s", flush=True)
    CACHE.put("trajectories", key, {"status": "complete", "payload": payload, "rows": rows})
    CACHE.delete_checkpoint(key)
    return rows

def kdb_rows(base, plans, *, common):
    rows = []
    for subset, (plan, y) in plans.items():
        started = perf_counter(); p, diag = predict_kdb_compiled(base, plan); seconds = perf_counter() - started
        contexts = sum(len(depths[min(base.selected_k_, len(depths)-1)]) for depths in base.tables_)
        rows.append({**common, "model_variant": "KDB", "learning_rate": np.nan, "order_seed": 0, "discriminative_pass": 0,
            "cumulative_updates": 0, "cumulative_training_seconds": base.training_seconds_, "optimizer": "generative", "subset": subset,
            **metric_record(y, p), **diag, "prediction_seconds": seconds, "prediction_sha256": _array_sha256(p),
            "discriminative_parameter_count": 0, "active_generative_contexts": contexts, "structure_sha256": _structure_sha256(base),
            "parameter_to_training_row_ratio": 0.0})
    return rows


## 8. Dataset execution with outer-test isolation and validation-only selection

`PRIMARY` evaluates every prespecified pass from 0–20 for fixed LR and each grid candidate, then chooses LR and best pass using inner-validation NLL only. `STRESS` reads the completed `PRIMARY` aggregate, selects up to three strongest directional overfitting configurations plus one low-degradation control, uses exactly nested per-class prefixes at 10/25/50/100%, and adds three deterministic row-order seeds only for selected `k>=2` cases. Test metrics never enter these selections.

The runner is restart- and memory-bounded: it checks for complete trajectory groups before preprocessing or structure construction; uses 32-bit factor indices when safe; allocates backoff score arrays only when needed; stores only one live optimizer state; evaluates test immediately when validation establishes a new best; deletes a checkpoint after its final trajectory JSON is committed; and explicitly releases per-k plans. A restarted PRIMARY run therefore skips completed splits before rebuilding their high-k structures.


In [ ]:
EXPECTED_SHARED_RAW_PATHS = {
    "Skin Segmentation": SHARED_RAW_CACHE / "openml" / "openml.org" / "api" / "v1" / "json" / "data" / "1502.gz",
    "MiniBooNE": SHARED_RAW_CACHE / "openml" / "openml.org" / "api" / "v1" / "json" / "data" / "41150.gz",
    "cod-rna": SHARED_RAW_CACHE / "cod-rna.train.libsvm", "Covertype": SHARED_RAW_CACHE / "covertype" / "samples_py3",
    "PokerHand": SHARED_RAW_CACHE / "uci-158.zip", "Localization": SHARED_RAW_CACHE / "uci-196.zip",
    "TwitterAbsoluteSigma": SHARED_RAW_CACHE / "Twitter-Absolute-Sigma-500.data", "CensusIncome": SHARED_RAW_CACHE / "uci-117.zip",
}

def find_primary_table_c():
    candidates = []
    for config in RESULT_CACHE_ROOT.glob("*/configuration.json"):
        try:
            meta = json.loads(config.read_text(encoding="utf-8")); table = config.parent / "exports" / "table_c_dataset_k_aggregate.csv"
            if meta.get("mode") == "PRIMARY" and table.exists(): candidates.append(table)
        except Exception: pass
    if not candidates: raise FileNotFoundError("STRESS requires a completed PRIMARY table C in this isolated cache")
    return pd.read_csv(max(candidates, key=lambda p: p.stat().st_mtime))

def stress_selection():
    table = find_primary_table_c(); table = table[table["learning_rate_role"].eq("validation-selected")].copy()
    anomalous = table.sort_values(["classical_overfitting_proportion", "mean_validation_degradation"], ascending=False).head(3)
    control = table.sort_values("mean_validation_degradation", ascending=True).head(1)
    selected = pd.concat([anomalous, control]).drop_duplicates(["dataset", "k", "model_variant"])
    return selected[["dataset", "k", "model_variant", "learning_rate"]].to_dict("records")

def unit_settings(k, specs):
    variants = [s["model_variant"] for s in specs if int(s["k"]) == k] if specs else MODEL_VARIANTS
    for variant in variants:
        if specs:
            rates = (float(next(s["learning_rate"] for s in specs if int(s["k"]) == k and s["model_variant"] == variant)),)
            seeds = DIAGNOSTIC_MODE["order_seeds"] if k >= 2 else (0,)
        else:
            rates = tuple(dict.fromkeys((FIXED_LEARNING_RATE,) + (LEARNING_RATE_GRID if DIAGNOSTIC_MODE["select_learning_rate"] else ())))
            seeds = (0,)
        for learning_rate in rates:
            for order_seed in seeds: yield variant, learning_rate, order_seed

def make_common_record(dataset_name, metadata, X, classes, split, k, fit_index, inner_val, outer_test, fraction):
    effective_k = min(int(k), X.shape[1] - 1)
    return {"configuration_id": CONFIGURATION_ID, "experiment_mode": EXPERIMENT_MODE, "dataset": dataset_name,
        "split_id": split["split_id"], "repeat": split["repeat"], "fold": split["fold"], "k": int(k), "effective_k": effective_k,
        "saturated_k": bool(k != effective_k), "rows": len(X), "features": X.shape[1], "classes": len(classes),
        "inner_fit_rows": len(fit_index), "inner_validation_rows": len(inner_val), "outer_test_rows": len(outer_test),
        "sample_fraction": fraction, "fit_index_sha256": _index_sha256(fit_index), "validation_index_sha256": _index_sha256(inner_val),
        "test_index_sha256": _index_sha256(outer_test), "preprocessing_fit_rows": len(fit_index),
        "outer_test_used_for_selection": False, "dataset_source": metadata.get("source"), "dataset_source_identity": metadata.get("source_identity")}

def completed_k_records(common, specs):
    records = []
    for variant, learning_rate, order_seed in unit_settings(int(common["k"]), specs):
        payload = trajectory_payload(common, learning_rate, variant, order_seed); key = CACHE.key(payload); cached = CACHE.get("trajectories", key)
        if cached is None: return False, []
        CACHE.delete_checkpoint(key); records.extend(cached["rows"])
    return True, records

def kdb_rows_from_completed_trajectory(records, common):
    candidates = [row for row in records if row["model_variant"] == "dKDB-generative-init" and
                  int(row["discriminative_pass"]) == 0 and np.isclose(float(row["learning_rate"]), FIXED_LEARNING_RATE)]
    if not candidates:
        candidates = [row for row in records if row["model_variant"] == "dKDB-generative-init" and int(row["discriminative_pass"]) == 0]
    rows = []
    for subset in ("train", "validation", "test"):
        source = next(row for row in candidates if row["subset"] == subset); row = dict(source)
        row.update(model_variant="KDB", learning_rate=np.nan, order_seed=0, optimizer="generative",
                   cumulative_updates=0, discriminative_parameter_count=0, parameter_to_training_row_ratio=0.0)
        rows.append(row)
    return rows

def run_experiment():
    trajectory_rows, baseline_rows, attempts, metadata_rows = [], [], [], []
    stress_specs = stress_selection() if EXPERIMENT_MODE == "STRESS" else None
    datasets_to_run = tuple(dict.fromkeys(item["dataset"] for item in stress_specs)) if stress_specs else DIAGNOSTIC_MODE["datasets"]
    missing = [str(EXPECTED_SHARED_RAW_PATHS[d]) for d in datasets_to_run if d in EXPECTED_SHARED_RAW_PATHS and not EXPECTED_SHARED_RAW_PATHS[d].exists()]
    if missing: raise FileNotFoundError("Missing shared raw inputs; existing cache is never modified:\n" + "\n".join(missing))
    for dataset_name in datasets_to_run:
        try:
            print(f"\nDATASET START {dataset_name}", flush=True)
            X, labels, metadata = load_dataset_cached(dataset_name, max_rows=DIAGNOSTIC_MODE["row_limits"].get(dataset_name), seed=RANDOM_SEED)
            X = _matrix(X); labels = np.asarray(labels, dtype=object); y, classes = _encode_global_labels(labels)
            metadata_rows.append({"dataset": dataset_name, "rows": len(X), "features": X.shape[1], "classes": len(classes),
                                  "source": metadata.get("source"), "source_identity": metadata.get("source_identity"),
                                  "paper_shape_matches": metadata.get("paper_shape_matches"), "processed_cache_hit": metadata.get("processed_cache_hit")})
            config = EvaluationConfig(rounds=DIAGNOSTIC_MODE["rounds"], folds=DIAGNOSTIC_MODE["folds"], random_state=RANDOM_SEED)
            plans = standard_split_plan(X, y, config)
            if DIAGNOSTIC_MODE["outer_split_limit"]: plans = plans[:DIAGNOSTIC_MODE["outer_split_limit"]]
            for split_number, split in enumerate(plans):
                print(f"SPLIT START {dataset_name} {split['split_id']}", flush=True)
                outer_train, outer_test = np.asarray(split["train_index"]), np.asarray(split["test_index"])
                inner_fit, inner_val = deterministic_inner_split(outer_train, y, split_number)
                assert not (set(inner_fit) & set(inner_val) or set(inner_fit) & set(outer_test) or set(inner_val) & set(outer_test))
                subsets = nested_stratified_subsets(inner_fit, y, DIAGNOSTIC_MODE["sample_fractions"], RANDOM_SEED + split_number)
                specs = [s for s in stress_specs if s["dataset"] == dataset_name] if stress_specs else None
                k_values = sorted(set(int(s["k"]) for s in specs)) if specs else DIAGNOSTIC_MODE["k_values"]
                for fraction, fit_index in subsets.items():
                    attempt_base = {"dataset": dataset_name, "split_id": split["split_id"], "sample_fraction": fraction}
                    try:
                        common_by_k = {int(k): make_common_record(dataset_name, metadata, X, classes, split, int(k), fit_index,
                                                                  inner_val, outer_test, fraction) for k in k_values}
                        cached_by_k = {int(k): completed_k_records(common_by_k[int(k)], specs) for k in k_values}
                        pending_k = [int(k) for k in k_values if not cached_by_k[int(k)][0]]
                        if not pending_k:
                            for k in k_values:
                                records = cached_by_k[int(k)][1]; trajectory_rows.extend(records)
                                baseline_rows.extend(kdb_rows_from_completed_trajectory(records, common_by_k[int(k)]))
                            attempts.append({**attempt_base, "status": "complete-cache-skip", "warning_error": None})
                            print(f"EARLY CACHE SKIP {dataset_name} {split['split_id']} fraction={fraction:g}: all k/variant/LR trajectories complete", flush=True)
                            continue
                        print(f"PREPROCESS {dataset_name} {split['split_id']} fraction={fraction:g}; pending k={pending_k}", flush=True)
                        preprocessor = MixedFoldDiscretizer(metadata["categorical_indices"]) if metadata.get("categorical_indices") else RuntimeOptimizedMDLDiscretizer()
                        preprocessor.fit(X[fit_index], y[fit_index]); assert preprocessor.fit_rows_ == len(fit_index)
                        X_fit = preprocessor.transform(X[fit_index]); X_val = preprocessor.transform(X[inner_val]); X_test = preprocessor.transform(X[outer_test])
                        y_fit, y_val, y_test = y[fit_index], y[inner_val], y[outer_test]
                        positive_pending = [k for k in pending_k if k > 0]
                        max_base = None
                        if positive_pending:
                            max_k = max(positive_pending); print(f"STRUCTURE {dataset_name} {split['split_id']} fitting shared max pending k={max_k}", flush=True)
                            max_base = RuntimeOptimizedKDBClassifier(k=max_k).fit(X_fit, y_fit)
                        zero_base = RuntimeOptimizedKDBClassifier(k=0).fit(X_fit, y_fit) if 0 in pending_k else None
                        for k in k_values:
                            common = common_by_k[int(k)]
                            if cached_by_k[int(k)][0]:
                                records = cached_by_k[int(k)][1]; trajectory_rows.extend(records)
                                baseline_rows.extend(kdb_rows_from_completed_trajectory(records, common))
                                print(f"K CACHE SKIP {dataset_name} {split['split_id']} k={k}: all trajectories complete", flush=True)
                                continue
                            base = _base_for_k(max_base, zero_base, k); effective_k = int(base.selected_k_)
                            print(f"PLAN {dataset_name} {split['split_id']} fraction={fraction:g} k={k}: compiling reusable prediction lookups", flush=True)
                            if effective_k != common["effective_k"]: raise AssertionError("Pre-cache effective-k prediction was wrong")
                            prediction_plans = {"train": (compile_prediction_plan(base, X_fit), y_fit),
                                                "validation": (compile_prediction_plan(base, X_val), y_val),
                                                "test": (compile_prediction_plan(base, X_test), y_test)}
                            baseline_rows.extend(kdb_rows(base, prediction_plans, common=common))
                            for variant, learning_rate, order_seed in unit_settings(int(k), specs):
                                trajectory_rows.extend(run_one_trajectory(base, X_fit, y_fit, prediction_plans, common=common,
                                                                         learning_rate=learning_rate, variant=variant, order_seed=order_seed))
                            del prediction_plans, base; gc.collect()
                        attempts.append({**attempt_base, "status": "complete", "warning_error": None})
                        del preprocessor, X_fit, X_val, X_test, y_fit, y_val, y_test, max_base, zero_base; gc.collect()
                    except Exception as exc:
                        attempts.append({**attempt_base, "status": "failed", "warning_error": f"{type(exc).__name__}: {exc}"}); print("FAILED", attempt_base, exc)
        except Exception as exc:
            attempts.append({"dataset": dataset_name, "split_id": None, "sample_fraction": None, "status": "failed", "warning_error": f"{type(exc).__name__}: {exc}"})
            print("FAILED DATASET", dataset_name, exc)
    return pd.DataFrame(trajectory_rows), pd.DataFrame(baseline_rows), pd.DataFrame(attempts), pd.DataFrame(metadata_rows)

if RUN_EXPERIMENT:
    TRAJECTORY_CANDIDATES, KDB_BASELINES, ATTEMPT_STATUS, DATASET_METADATA = run_experiment()
else:
    cached = CACHE.collect("trajectories"); TRAJECTORY_CANDIDATES = pd.DataFrame(sum(cached.get("rows", pd.Series(dtype=object)).tolist(), [])) if not cached.empty else pd.DataFrame()
    KDB_BASELINES = pd.DataFrame(); ATTEMPT_STATUS = pd.DataFrame(); DATASET_METADATA = pd.DataFrame()

if EXPERIMENT_MODE == "SMOKE":
    assert len(ATTEMPT_STATUS) and ATTEMPT_STATUS["status"].str.startswith("complete").all(), ATTEMPT_STATUS
    assert set(TRAJECTORY_CANDIDATES["dataset"]) == {"Skin Segmentation", "cod-rna"}
display(ATTEMPT_STATUS); display(TRAJECTORY_CANDIDATES.head())


## 9. Validation-selected learning rates and precise trajectory summaries


In [ ]:
def attach_learning_rate_roles(frame):
    if frame.empty: return frame
    keys = ["dataset", "split_id", "k", "model_variant", "sample_fraction", "order_seed"]
    output = []
    for _, group in frame.groupby(keys, dropna=False, sort=False):
        fixed = group[np.isclose(group["learning_rate"], FIXED_LEARNING_RATE)].copy(); fixed["learning_rate_role"] = "fixed-0.01"; output.append(fixed)
        if DIAGNOSTIC_MODE["select_learning_rate"]:
            validation = group[group["subset"].eq("validation")]
            choice = (validation.groupby("learning_rate", as_index=False)["nll"].min().sort_values(["nll", "learning_rate"]).iloc[0]["learning_rate"])
            selected = group[np.isclose(group["learning_rate"], choice)].copy(); selected["learning_rate_role"] = "validation-selected"; output.append(selected)
    return pd.concat(output, ignore_index=True) if output else pd.DataFrame()

TRAJECTORY_RESULTS = attach_learning_rate_roles(TRAJECTORY_CANDIDATES)

def at(frame, subset, d, metric):
    value = frame[(frame["subset"] == subset) & (frame["discriminative_pass"] == d)][metric]
    return float(value.iloc[0]) if len(value) else np.nan

def summarize_trajectory(frame):
    val = frame[frame["subset"].eq("validation")].sort_values("discriminative_pass"); passes = val["discriminative_pass"].to_numpy(int)
    best = int(val.loc[val["nll"].idxmin(), "discriminative_pass"]); final = int(passes.max())
    val_values = val.set_index("discriminative_pass")["nll"]
    onset = np.nan
    for d in passes[passes > best]:
        tail = [x for x in passes if x >= d][:2]
        if len(tail) == 2 and all(val_values[x] > val_values[best] for x in tail): onset = int(d); break
    train_best, train_final = at(frame, "train", best, "nll"), at(frame, "train", final, "nll")
    val_best, val_final = at(frame, "validation", best, "nll"), at(frame, "validation", final, "nll")
    test_best, test_final = at(frame, "test", best, "nll"), at(frame, "test", final, "nll")
    row = {c: frame.iloc[0][c] for c in ["dataset", "split_id", "repeat", "fold", "k", "effective_k", "model_variant", "learning_rate",
        "learning_rate_role", "optimizer", "rows", "features", "classes", "inner_fit_rows", "inner_validation_rows", "outer_test_rows",
        "sample_fraction", "order_seed", "structure_sha256", "discriminative_parameter_count", "active_generative_contexts", "parameter_to_training_row_ratio"]}
    row.update(best_validation_pass=best, first_sustained_deterioration_pass=onset, final_pass=final,
        initial_train_nll=at(frame,"train",0,"nll"), initial_validation_nll=at(frame,"validation",0,"nll"), initial_test_nll=at(frame,"test",0,"nll"),
        d1_train_nll=at(frame,"train",1,"nll"), d1_validation_nll=at(frame,"validation",1,"nll"), d1_test_nll=at(frame,"test",1,"nll"),
        best_train_nll=train_best, best_validation_nll=val_best, best_test_nll=test_best,
        d10_train_nll=at(frame,"train",10,"nll"), d10_validation_nll=at(frame,"validation",10,"nll"), d10_test_nll=at(frame,"test",10,"nll"),
        final_train_nll=train_final, final_validation_nll=val_final, final_test_nll=test_final,
        train_divergence=train_final-train_best, validation_degradation=val_final-val_best, test_degradation=test_final-test_best,
        generalisation_gap_best=val_best-train_best, generalisation_gap_final=val_final-train_final,
        gap_growth=(val_final-train_final)-(val_best-train_best), d1_minus_best_validation_nll=at(frame,"validation",1,"nll")-val_best,
        d10_minus_best_validation_nll=at(frame,"validation",10,"nll")-val_best if final >= 10 else np.nan,
        d1_train_rmse=at(frame,"train",1,"rmse"), d1_validation_rmse=at(frame,"validation",1,"rmse"), d1_test_rmse=at(frame,"test",1,"rmse"),
        d10_train_rmse=at(frame,"train",10,"rmse"), d10_validation_rmse=at(frame,"validation",10,"rmse"), d10_test_rmse=at(frame,"test",10,"rmse"),
        best_train_rmse=at(frame,"train",best,"rmse"), best_validation_rmse=at(frame,"validation",best,"rmse"), best_test_rmse=at(frame,"test",best,"rmse"),
        final_train_rmse=at(frame,"train",final,"rmse"), final_validation_rmse=at(frame,"validation",final,"rmse"), final_test_rmse=at(frame,"test",final,"rmse"),
        final_weight_l1=at(frame,"validation",final,"weight_l1"), final_weight_l2=at(frame,"validation",final,"weight_l2"),
        final_weight_max_abs=at(frame,"validation",final,"weight_max_abs"), final_backoff_rate=at(frame,"test",final,"backoff_rate"),
        final_backoff_corrections=at(frame,"test",final,"backoff_corrections"), context_support_median=at(frame,"validation",final,"context_support_median"),
        context_support_mean=at(frame,"validation",final,"context_support_mean"))
    row["validation_rmse_degradation"] = row["final_validation_rmse"] - row["best_validation_rmse"]
    row["test_rmse_degradation"] = row["final_test_rmse"] - row["best_test_rmse"]
    row["train_rmse_divergence"] = row["final_train_rmse"] - row["best_train_rmse"]
    row["rmse_gap_growth"] = ((row["final_validation_rmse"]-row["final_train_rmse"]) -
                              (row["best_validation_rmse"]-row["best_train_rmse"]))
    row["classical_overfitting_pattern"] = bool(row["train_divergence"] <= 0 and row["validation_degradation"] > 0 and row["gap_growth"] > 0 and row["test_degradation"] > 0)
    return row

summary_keys = ["dataset", "split_id", "k", "model_variant", "learning_rate_role", "sample_fraction", "order_seed"]
TABLE_B_TRAJECTORY_SUMMARY = pd.DataFrame([summarize_trajectory(g) for _, g in TRAJECTORY_RESULTS.groupby(summary_keys, dropna=False, sort=False)]) if not TRAJECTORY_RESULTS.empty else pd.DataFrame()

if not TABLE_B_TRAJECTORY_SUMMARY.empty:
    agg_keys = ["dataset", "k", "model_variant", "learning_rate_role"]
    TABLE_C_DATASET_K_AGGREGATE = TABLE_B_TRAJECTORY_SUMMARY.groupby(agg_keys, as_index=False).agg(
        learning_rate=("learning_rate", "mean"), mean_best_validation_pass=("best_validation_pass", "mean"), sd_best_validation_pass=("best_validation_pass", "std"),
        mean_validation_degradation=("validation_degradation", "mean"), mean_test_degradation=("test_degradation", "mean"), mean_gap_growth=("gap_growth", "mean"),
        splits_showing_classical_pattern=("classical_overfitting_pattern", "sum"), outer_splits=("split_id", "nunique"),
        classical_overfitting_proportion=("classical_overfitting_pattern", "mean"), mean_d1_minus_best=("d1_minus_best_validation_nll", "mean"),
        mean_d10_minus_best=("d10_minus_best_validation_nll", "mean"))
else: TABLE_C_DATASET_K_AGGREGATE = pd.DataFrame()

TABLE_D_SAMPLE_SIZE_STRESS = TABLE_B_TRAJECTORY_SUMMARY[TABLE_B_TRAJECTORY_SUMMARY["sample_fraction"].lt(1)].copy() if not TABLE_B_TRAJECTORY_SUMMARY.empty else pd.DataFrame()
if not TABLE_D_SAMPLE_SIZE_STRESS.empty:
    TABLE_D_SAMPLE_SIZE_STRESS["parameters_per_row"] = TABLE_D_SAMPLE_SIZE_STRESS["discriminative_parameter_count"] / TABLE_D_SAMPLE_SIZE_STRESS["inner_fit_rows"]
TABLE_E_OPTIMISATION_SENSITIVITY = TABLE_B_TRAJECTORY_SUMMARY[[c for c in ["dataset","k","model_variant","learning_rate","order_seed","final_train_nll","best_validation_nll","final_validation_nll","best_test_nll","final_weight_l2"] if c in TABLE_B_TRAJECTORY_SUMMARY.columns]].copy()

# Join exact KDB baselines into Table B without using test results for selection.
if not TABLE_B_TRAJECTORY_SUMMARY.empty and not KDB_BASELINES.empty:
    piv = KDB_BASELINES.pivot_table(index=["dataset","split_id","k","sample_fraction"], columns="subset", values=["nll","rmse"], aggfunc="first")
    piv.columns = [f"kdb_{subset}_{metric}" for metric, subset in piv.columns]; piv = piv.reset_index()
    TABLE_B_TRAJECTORY_SUMMARY = TABLE_B_TRAJECTORY_SUMMARY.merge(piv, on=["dataset","split_id","k","sample_fraction"], how="left")
    TABLE_B_TRAJECTORY_SUMMARY["final_minus_kdb_validation_nll"] = TABLE_B_TRAJECTORY_SUMMARY["final_validation_nll"] - TABLE_B_TRAJECTORY_SUMMARY["kdb_validation_nll"]

display(TABLE_B_TRAJECTORY_SUMMARY); display(TABLE_C_DATASET_K_AGGREGATE)


## 10. Publication-readable figures


In [ ]:
EXPORT_DIRECTORY = EXPERIMENT_DIRECTORY / "exports"; FIGURE_DIRECTORY = EXPERIMENT_DIRECTORY / "figures"
EXPORT_DIRECTORY.mkdir(parents=True, exist_ok=True); FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)

def savefig(fig, stem):
    fig.savefig(FIGURE_DIRECTORY / f"{stem}.png", dpi=200, bbox_inches="tight"); fig.savefig(FIGURE_DIRECTORY / f"{stem}.pdf", bbox_inches="tight"); plt.close(fig)

def trajectory_figures(frame, metric):
    for (dataset, variant, role), group in frame.groupby(["dataset","model_variant","learning_rate_role"], sort=False):
        ks = sorted(group["k"].unique()); fig, axes = plt.subplots(math.ceil(len(ks)/2), 2, figsize=(13, 4.2*math.ceil(len(ks)/2)), squeeze=False)
        for ax, k in zip(axes.ravel(), ks):
            g = group[group["k"].eq(k)]
            mean = g.groupby(["discriminative_pass","subset"], as_index=False)[metric].mean()
            for subset, color in [("train","#1f77b4"),("validation","#d62728"),("test","#2ca02c")]:
                s = mean[mean["subset"].eq(subset)]; ax.plot(s["discriminative_pass"], s[metric], marker="o", ms=3, label=subset, color=color)
            kdb = KDB_BASELINES[(KDB_BASELINES["dataset"]==dataset)&(KDB_BASELINES["k"]==k)&(KDB_BASELINES["subset"]=="validation")]
            if len(kdb): ax.axhline(kdb[metric].mean(), color="black", ls="--", lw=1.2, label="KDB validation")
            best = int(mean[mean["subset"].eq("validation")].sort_values(metric).iloc[0]["discriminative_pass"]); ax.axvline(best, color="#9467bd", ls=":")
            ax.set(title=f"k={k}; best validation pass={best}", xlabel="discriminative pass", ylabel=metric.upper()); ax.legend(fontsize=8)
        for ax in axes.ravel()[len(ks):]: ax.axis("off")
        fig.suptitle(f"{dataset} — {variant} — {role}"); fig.tight_layout(); savefig(fig, f"{metric}_{dataset}_{variant}_{role}".replace(" ","_").replace("/","_"))

def heatmaps(frame):
    for (dataset, variant, role), group in frame.groupby(["dataset","model_variant","learning_rate_role"], sort=False):
        means = group.groupby(["k","discriminative_pass","subset"], as_index=False)["nll"].mean()
        train = means[means.subset.eq("train")].pivot(index="k",columns="discriminative_pass",values="nll")
        val = means[means.subset.eq("validation")].pivot(index="k",columns="discriminative_pass",values="nll").reindex_like(train)
        initial = val.iloc[:,0]
        for values, title, stem in [(val-train,"Validation NLL - train NLL","generalisation_gap"),(val.sub(initial,axis=0),"Validation NLL change from initial","validation_change")]:
            fig, ax = plt.subplots(figsize=(10, max(3, .55*len(values)))); image=ax.imshow(values,aspect="auto",cmap="coolwarm")
            ax.set(yticks=range(len(values.index)),yticklabels=values.index,xticks=range(len(values.columns)),xticklabels=values.columns,xlabel="pass",ylabel="k",title=f"{dataset}: {title}")
            fig.colorbar(image,ax=ax); fig.tight_layout(); savefig(fig,f"{stem}_{dataset}_{variant}_{role}".replace(" ","_"))

if not TRAJECTORY_RESULTS.empty:
    trajectory_figures(TRAJECTORY_RESULTS, "nll"); trajectory_figures(TRAJECTORY_RESULTS, "rmse"); heatmaps(TRAJECTORY_RESULTS)
if not TABLE_C_DATASET_K_AGGREGATE.empty:
    for dataset, group in TABLE_C_DATASET_K_AGGREGATE.groupby("dataset"):
        fig, ax = plt.subplots(figsize=(8,4.5))
        for (variant,role), g in group.groupby(["model_variant","learning_rate_role"]): ax.plot(g["k"],g["mean_best_validation_pass"],marker="o",label=f"{variant}; {role}")
        ax.set(xlabel="k",ylabel="mean best validation pass",title=f"{dataset}: best pass vs k"); ax.legend(fontsize=8); fig.tight_layout(); savefig(fig,f"best_pass_{dataset}".replace(" ","_"))
if not TABLE_D_SAMPLE_SIZE_STRESS.empty:
    for dataset, group in TABLE_D_SAMPLE_SIZE_STRESS.groupby("dataset"):
        fig, ax = plt.subplots(figsize=(8,4.5))
        for k,g in group.groupby("k"): ax.plot(g["inner_fit_rows"],g["validation_degradation"],marker="o",label=f"k={k}")
        ax.set_xscale("log"); ax.set(xlabel="training rows",ylabel="validation degradation",title=f"{dataset}: size stress"); ax.legend(); fig.tight_layout(); savefig(fig,f"size_stress_{dataset}".replace(" ","_"))
if not TABLE_B_TRAJECTORY_SUMMARY.empty:
    fig, ax = plt.subplots(figsize=(8,5)); sc=ax.scatter(TABLE_B_TRAJECTORY_SUMMARY["parameter_to_training_row_ratio"],TABLE_B_TRAJECTORY_SUMMARY["validation_degradation"],c=TABLE_B_TRAJECTORY_SUMMARY["k"],cmap="viridis",alpha=.75)
    ax.set(xlabel="discriminative parameters / training row",ylabel="validation degradation",title="Parameter complexity and held-out deterioration"); fig.colorbar(sc,ax=ax,label="k"); fig.tight_layout(); savefig(fig,"parameter_complexity")
print("Figures:", FIGURE_DIRECTORY.resolve())


## 11. Required tables, attempt ledger, and machine-readable exports


In [ ]:
def atomic_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True); tmp=path.with_suffix(path.suffix+".part"); frame.to_csv(tmp,index=False); tmp.replace(path)
exports = {
    "table_a_implementation_audit.csv": TABLE_A_IMPLEMENTATION_AUDIT,
    "table_b_trajectory_summary.csv": TABLE_B_TRAJECTORY_SUMMARY,
    "table_c_dataset_k_aggregate.csv": TABLE_C_DATASET_K_AGGREGATE,
    "table_d_sample_size_stress.csv": TABLE_D_SAMPLE_SIZE_STRESS,
    "table_e_optimisation_sensitivity.csv": TABLE_E_OPTIMISATION_SENSITIVITY,
    "trajectory_results_long.csv": TRAJECTORY_RESULTS, "trajectory_candidates_long.csv": TRAJECTORY_CANDIDATES,
    "kdb_baselines.csv": KDB_BASELINES, "attempt_status.csv": ATTEMPT_STATUS, "dataset_metadata.csv": DATASET_METADATA,
    "preflight_checks.csv": PREFLIGHT_CHECKS,
}
for filename, frame in exports.items(): atomic_csv(frame, EXPORT_DIRECTORY / filename)
display(TABLE_A_IMPLEMENTATION_AUDIT); display(TABLE_D_SAMPLE_SIZE_STRESS); display(TABLE_E_OPTIMISATION_SENSITIVITY.head(30)); display(ATTEMPT_STATUS)
print("Exports:", EXPORT_DIRECTORY.resolve())


## 12. Automatic diagnostic interpretation (evidence, not verdict inflation)


In [ ]:
def automatic_summary():
    scope = "SMOKE is a pipeline validation, not a dataset-wide scientific conclusion." if EXPERIMENT_MODE == "SMOKE" else "Results summarize the configured completed runs."
    selected_role = "validation-selected" if "validation-selected" in set(TABLE_B_TRAJECTORY_SUMMARY.get("learning_rate_role", [])) else "fixed-0.01"
    s = TABLE_B_TRAJECTORY_SUMMARY[TABLE_B_TRAJECTORY_SUMMARY.get("learning_rate_role", pd.Series(dtype=str)).eq(selected_role)].copy()
    classical = s[s.get("classical_overfitting_pattern", False).eq(True)] if len(s) else s
    where = ", ".join(f"{r.dataset}/k={int(r.k)}/{r.model_variant}" for r in classical.itertuples()) or "none in completed configurations"
    best = float(s["best_validation_pass"].mean()) if len(s) else np.nan
    higher_k = s.groupby("k")["gap_growth"].mean().to_dict() if len(s) else {}
    lr_answer = "not run in SMOKE" if not DIAGNOSTIC_MODE["select_learning_rate"] else "reported by paired fixed-0.01 and validation-selected rows in Tables B/C"
    stress_answer = "not run; switch to STRESS after PRIMARY" if EXPERIMENT_MODE != "STRESS" else "reported in Table D and size-stress figures"
    order_answer = "not triggered in this mode" if EXPERIMENT_MODE != "STRESS" else "reported in Table E for selected k>=2 configurations"
    lines = [
        f"**Scope:** {scope}",
        f"1. Direct overfitting signature: {len(classical)} of {len(s)} completed validation-selected/fixed trajectories satisfy the directional pattern; magnitudes are in Table B.",
        f"2. Dataset/k locations: {where}.",
        f"3. Mean best held-out pass: {best:.2f} (do not treat this mean as a universal stopping rule).",
        "4. D=1 status: see `d1_minus_best_validation_nll`; zero means optimal on the discrete checkpoint grid, positive means worse than the validation-selected pass.",
        "5. D=10 status: see `d10_minus_best_validation_nll`; it is unavailable by design in SMOKE (D<=3).",
        f"6. Higher-k pattern: mean gap growth by k is {higher_k}; order sensitivity is assessed separately rather than folded into an overfitting label.",
        f"7. Training-size effect: {stress_answer}.",
        "8. Complexity/context association: inspect Table B and the parameter-complexity plot; association is descriptive and not causal.",
        f"9. Learning-rate control: {lr_answer}.",
        f"10. Streaming-order sensitivity: {order_answer}.",
        "11. Initialization audit: repository trained dKDB starts at w=0; the paper studies this and w=1. Only w=1 reproduces KDB immediately before update 1.",
        "12. Initialization impact: compare model variants at matched split/k/LR; the notebook never silently substitutes one for the other.",
        "13. Overall diagnosis: classify individual rows using trajectory direction, KDB comparison, LR/order sensitivity, and stress results; poor final test performance alone is not called overfitting.",
        "\nDiagnostic labels are non-exclusive: A overfitting-consistent; B under-optimisation; C optimisation instability; D shared high-k/KDB issue; E dKDB-specific deterioration; F implementation-semantic discrepancy."
    ]
    return "\n\n".join(lines)

DIAGNOSTIC_SUMMARY = automatic_summary(); display(Markdown(DIAGNOSTIC_SUMMARY))
(EXPORT_DIRECTORY / "diagnostic_summary.md").write_text(DIAGNOSTIC_SUMMARY, encoding="utf-8")


## 13. Final readiness assertions


In [ ]:
assert REFERENCE_CODE_SHA256 == REFERENCE_CODE_SHA256_EXPECTED
assert PREFLIGHT_CHECKS["status"].eq("PASS").all()
assert not TRAJECTORY_RESULTS.empty
assert TRAJECTORY_RESULTS["outer_test_used_for_selection"].eq(False).all()
assert (TRAJECTORY_RESULTS["preprocessing_fit_rows"] == TRAJECTORY_RESULTS["inner_fit_rows"]).all()
for _, group in TRAJECTORY_RESULTS.groupby(["dataset","split_id","k","model_variant","learning_rate","sample_fraction","order_seed"]):
    expected_full = set(range(int(DIAGNOSTIC_MODE["max_pass"]) + 1))
    assert set(group[group["subset"].eq("train")]["discriminative_pass"]) == expected_full
    assert set(group[group["subset"].eq("validation")]["discriminative_pass"]) == expected_full
    validation = group[group["subset"].eq("validation")]
    best_pass = int(validation.loc[validation["nll"].idxmin(), "discriminative_pass"])
    required_test = {d for d in TEST_CHECKPOINTS if d <= DIAGNOSTIC_MODE["max_pass"]} | {best_pass}
    assert required_test.issubset(set(group[group["subset"].eq("test")]["discriminative_pass"]))
    assert group["generative_theta_sha256"].nunique() == 1
assert TRAJECTORY_RESULTS.groupby(["dataset","split_id","k"])["structure_sha256"].nunique().max() == 1
assert np.isfinite(TRAJECTORY_RESULTS[["nll","rmse","classification_error"]].to_numpy()).all()
if EXPERIMENT_MODE == "SMOKE":
    assert TRAJECTORY_RESULTS["discriminative_pass"].max() == 3
    assert set(TRAJECTORY_RESULTS["k"]) == {1,5}
    assert ATTEMPT_STATUS["status"].str.startswith("complete").all()
print("READY — fresh-kernel execution, audit assertions, split/preprocessing contracts, trajectories, tables, figures, and exports all completed.")
print("PRIMARY was not launched." if EXPERIMENT_MODE == "SMOKE" else f"Completed configured {EXPERIMENT_MODE} run.")


# FINAL PLOTS 
Chrome crashed mid-run so plotting from cached results

In [ ]:
from pathlib import Path
from IPython.display import display, Image, Markdown

figure_dir = Path(
    r"C:\Users\anagh\Comparative Study\Limited-Pass-Learning---A-Comparative-Study"
    r"\dkdb overfitting diagnostic cache\results\2e219c54a929d16f0cc7\figures"
)

for path in sorted(figure_dir.glob("*.png")):
    display(Markdown(f"### {path.stem}"))
    display(Image(filename=str(path)))